In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
# Importação do dataset
df = pd.read_csv("mecaniqa_dataset.csv")

# Preparação dos dados
df.columns = df.columns.str.strip()

df["Data"] = pd.to_datetime(
    df["Data"],
    format="%Y-%m-%d",
    errors="raise"
)

df = df.set_index("Data")
df = df.sort_index()

In [4]:
# Criação da base usada pelo modelo
base_modelo = pd.DataFrame(index=df.index)

# Usa somente informações dos dias anteriores
base_modelo["Trocas_Oleo_Ontem"] = df["Trocas_Oleo"].shift(1)
base_modelo["Trocas_Oleo_7_Dias_Atras"] = df["Trocas_Oleo"].shift(7)
base_modelo["Media_Ultimos_7_Dias"] = (
    df["Trocas_Oleo"]
    .shift(1)
    .rolling(window=7)
    .mean()
)
base_modelo["Dia_Da_Semana"] = base_modelo.index.dayofweek

# Valor que o modelo deverá prever
base_modelo["Alvo"] = df["Trocas_Oleo"]

# Remove apenas as linhas sem o valor alvo
base_modelo = base_modelo.dropna(subset=["Alvo"])

display(base_modelo.head(10))

,Trocas_Oleo_Ontem,Trocas_Oleo_7_Dias_Atras,Media_Ultimos_7_Dias,Dia_Da_Semana,Alvo
Data,,,,,
2024-01-01,NaN,NaN,NaN,0,11.0
2024-01-02,11.0,NaN,NaN,1,9.0
2024-01-03,9.0,NaN,NaN,2,12.0
2024-01-04,12.0,NaN,NaN,3,15.0
2024-01-05,15.0,NaN,NaN,4,25.0
2024-01-06,25.0,NaN,NaN,5,25.0
2024-01-07,25.0,NaN,NaN,6,31.0
2024-01-08,31.0,11.0,18.285714,0,13.0
2024-01-09,13.0,9.0,18.571429,1,10.0


In [5]:
# Separação entre variáveis de entrada e alvo
X = base_modelo.drop(columns="Alvo")
y = base_modelo["Alvo"]

# Divisão cronológica: 80% para treino e 20% para teste
indice_corte = int(len(base_modelo) * 0.80)

X_train = X.iloc[:indice_corte]
X_test = X.iloc[indice_corte:]

y_train = y.iloc[:indice_corte]
y_test = y.iloc[indice_corte:]

print("Registros de treino:", len(X_train))
print("Registros de teste:", len(X_test))

Registros de treino: 580
Registros de teste: 145


In [6]:
# Construção do Pipeline
pipeline = Pipeline(
    steps=[
        ("preencher_nulos", SimpleImputer(strategy="median")),
        ("padronizar_escala", StandardScaler()),
        ("modelo", Ridge(alpha=1.0))
    ]
)

# Treinamento em uma única linha
pipeline.fit(X_train, y_train)

print("Pipeline treinado com sucesso!")

Pipeline treinado com sucesso!


In [7]:
# Avaliação do modelo
previsoes_teste = pipeline.predict(X_test)
erro_medio = mean_absolute_error(y_test, previsoes_teste)

print(f"Erro absoluto médio: {erro_medio:.2f} trocas de óleo\n")

# Previsão para o próximo dia
serie_oleo = df["Trocas_Oleo"].copy()
serie_preenchida = (
    serie_oleo
    .interpolate(method="time")
    .ffill()
    .bfill()
)

proxima_data = df.index.max() + pd.Timedelta(days=1)

dados_proximo_dia = pd.DataFrame(
    {
        "Trocas_Oleo_Ontem": [serie_preenchida.iloc[-1]],
        "Trocas_Oleo_7_Dias_Atras": [serie_preenchida.iloc[-7]],
        "Media_Ultimos_7_Dias": [serie_preenchida.iloc[-7:].mean()],
        "Dia_Da_Semana": [proxima_data.dayofweek]
    },
    index=[proxima_data]
)

previsao_proximo_dia = pipeline.predict(dados_proximo_dia)[0]
previsao_proximo_dia = max(0, previsao_proximo_dia)

print("Data da previsão:", proxima_data.date())
print(f"Previsão: {previsao_proximo_dia:.0f} trocas de óleo")

Erro absoluto médio: 3.62 trocas de óleo

Data da previsão: 2026-01-01
Previsão: 32 trocas de óleo
